# Atmospheric Reentry Simulation

This notebook demonstrates atmospheric reentry physics using the multi-body atmosphere model.

## Topics Covered
1. Atmospheric density profiles (Earth, Mars, Venus)
2. Drag force calculations
3. Heat flux estimation
4. Reentry trajectory simulation
5. SOI-based automatic atmosphere detection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../space_state')

from base import (
    Propagator, Spacecraft, Thruster, BODIES,
    get_circular_state, get_body_position, get_body_velocity,
    MultiBodyAtmosphere, compute_drag_acceleration, ATMOSPHERE_PARAMS,
    create_altitude_event, create_impact_event,
)
from base.star_structures import PropagationResult
from base.star_constants import IDX_R, IDX_V, IDX_M, AU_TO_KM

## 1. Atmospheric Density Profiles

Compare atmosphere models for Earth, Mars, and Venus.

In [ ]:
# Create atmosphere model
atm = MultiBodyAtmosphere()

# Altitude range
altitudes = np.linspace(0, 500, 500)

# Get densities for each body
bodies = ['earth', 'mars barycenter', 'venus barycenter']
densities = {}

for body in bodies:
    densities[body] = [atm.density(body, alt) for alt in altitudes]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax = axes[0]
for body in bodies:
    ax.plot(altitudes, densities[body], label=body.replace(' barycenter', '').title())
ax.set_xlabel('Altitude (km)')
ax.set_ylabel('Density (kg/m³)')
ax.set_title('Atmospheric Density (Linear Scale)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100)

# Log scale (full range)
ax = axes[1]
for body in bodies:
    ax.semilogy(altitudes, densities[body], label=body.replace(' barycenter', '').title())
ax.set_xlabel('Altitude (km)')
ax.set_ylabel('Density (kg/m³)')
ax.set_title('Atmospheric Density (Log Scale)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(1e-15, 1e3)

plt.tight_layout()
plt.show()

# Surface densities
print("Surface Densities:")
print("=" * 40)
for body in bodies:
    name = body.replace(' barycenter', '').title()
    rho = atm.density(body, 0.0)
    print(f"{name}: {rho:.4f} kg/m³")

## 2. Heat Flux Analysis

Calculate convective heating during reentry using the Sutton-Graves correlation.

In [ ]:
# Heat flux vs altitude for different velocities
velocities_kmps = [5, 7, 10, 12]  # km/s
altitudes = np.linspace(40, 150, 100)

fig, ax = plt.subplots(figsize=(10, 6))

for v_kmps in velocities_kmps:
    v_mps = v_kmps * 1000  # Convert to m/s
    heat_flux = [atm.heat_flux(alt, v_mps, 'earth') / 1e6 for alt in altitudes]  # MW/m²
    ax.plot(altitudes, heat_flux, label=f'{v_kmps} km/s', linewidth=2)

ax.set_xlabel('Altitude (km)', fontsize=12)
ax.set_ylabel('Heat Flux (MW/m²)', fontsize=12)
ax.set_title('Convective Heat Flux vs Altitude (Earth Reentry)', fontsize=14)
ax.legend(title='Entry velocity')
ax.grid(True, alpha=0.3)
ax.set_xlim(40, 150)

plt.tight_layout()
plt.show()

# Peak heating conditions
print("\nPeak Heating Analysis (80 km altitude):")
print("=" * 50)
for v_kmps in velocities_kmps:
    q = atm.heat_flux(80, v_kmps * 1000, 'earth')
    print(f"V = {v_kmps} km/s: {q/1e6:.2f} MW/m²")

## 3. Drag Force Calculations

Calculate drag deceleration vs altitude.

In [ ]:
# Capsule parameters
mass = 5000  # kg
area = 12.0  # m² (cross-sectional)
Cd = 1.2     # Drag coefficient

# Calculate drag deceleration
altitudes = np.linspace(40, 400, 100)
v = 7.8  # km/s (LEO orbital velocity)
v_mps = v * 1000

decelerations = []
for alt in altitudes:
    rho = atm.density('earth', alt)
    F_drag = 0.5 * rho * v_mps**2 * Cd * area
    a = F_drag / mass
    decelerations.append(a)

# Convert to g's
g0 = 9.81
decelerations_g = np.array(decelerations) / g0

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(altitudes, decelerations_g, 'r-', linewidth=2)
ax.axhline(y=10, color='orange', linestyle='--', label='10g limit (typical)')
ax.axhline(y=1, color='green', linestyle='--', label='1g')

ax.set_xlabel('Altitude (km)', fontsize=12)
ax.set_ylabel('Deceleration (g)', fontsize=12)
ax.set_title(f'Aerodynamic Deceleration (V={v} km/s, m={mass} kg)', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3, which='both')
ax.set_ylim(1e-4, 1e3)

plt.tight_layout()
plt.show()

# Find peak g-load altitude
peak_idx = np.argmax(decelerations_g)
print(f"\nPeak g-load: {decelerations_g[peak_idx]:.1f}g at {altitudes[peak_idx]:.0f} km altitude")

## 4. Reentry Trajectory Simulation

Simulate a deorbit and reentry trajectory with atmospheric drag enabled.

In [ ]:
# Create reentry vehicle
reentry_vehicle = Spacecraft(
    name="Reentry Capsule",
    dry_mass=5000,
    propellant_mass=100,  # Small amount for deorbit burn
    thrusters=[Thruster("Deorbit", thrust_n=4400, isp_s=220)],
)

# Start in LEO (lower altitude for faster reentry)
altitude_km = 200
t0 = 0.0

# Get circular orbit state
state = get_circular_state(
    body='earth',
    altitude_km=altitude_km,
    inclination_deg=28.5,
    time=t0,
    mass=reentry_vehicle.mass,
)

# Apply retrograde delta-V to lower perigee into atmosphere
# Typical deorbit burn: ~100 m/s retrograde
r_earth = get_body_position('earth', t0)
v_earth = get_body_velocity('earth', t0)
v_rel = state[3:6] - v_earth
v_hat = v_rel / np.linalg.norm(v_rel)

deorbit_dv_kmps = 0.15  # 150 m/s
deorbit_dv_au_day = -deorbit_dv_kmps / 1731.46  # retrograde
state[3:6] += deorbit_dv_au_day * v_hat

print(f"Initial altitude: {altitude_km} km")
print(f"Deorbit delta-V: {deorbit_dv_kmps*1000:.0f} m/s retrograde")

In [ ]:
# Set up propagator with drag enabled
result = PropagationResult()
result.initialize(state, t0)

propagator = Propagator(
    ship=reentry_vehicle,
    guidance=None,  # No guidance needed
    result=result,
    drag_enabled=True,
    drag_area=12.0,  # m²
    drag_coefficient=1.2,
)

# Create impact event to stop at Earth surface
impact_event = create_impact_event('earth')

# Propagate until impact or 2 hours
t_end = 2.0 / 24  # 2 hours in days
propagator.propagate(t_end, events=[impact_event], max_step=1e-4)

print(f"\nSimulation complete")
print(f"Final time: {result.t[-1] * 24 * 60:.1f} minutes")
print(f"Time steps: {len(result.t)}")

## 5. Analyze Reentry Trajectory

In [ ]:
# Calculate altitude profile
r_earth = get_body_position('earth', 0.0)
earth_radius_km = 6371.0

positions = result.y[:3, :].T
velocities = result.y[3:6, :].T

# Calculate altitude and velocity magnitude
altitudes_km = []
velocities_kmps = []

for i in range(len(result.t)):
    r = positions[i]
    v = velocities[i]
    r_earth_t = get_body_position('earth', result.t[i])
    v_earth_t = get_body_velocity('earth', result.t[i])
    
    dist_km = np.linalg.norm(r - r_earth_t) * AU_TO_KM
    alt_km = dist_km - earth_radius_km
    altitudes_km.append(alt_km)
    
    v_rel = v - v_earth_t
    v_kmps = np.linalg.norm(v_rel) * 1731.46
    velocities_kmps.append(v_kmps)

altitudes_km = np.array(altitudes_km)
velocities_kmps = np.array(velocities_kmps)
times_min = result.t * 24 * 60

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Altitude vs time
ax = axes[0, 0]
ax.plot(times_min, altitudes_km, 'b-', linewidth=2)
ax.axhline(y=100, color='red', linestyle='--', label='Karman line (100 km)')
ax.axhline(y=0, color='brown', linestyle='-', label='Earth surface')
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('Altitude (km)')
ax.set_title('Altitude vs Time')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-50, max(altitudes_km) * 1.1)

# Velocity vs time
ax = axes[0, 1]
ax.plot(times_min, velocities_kmps, 'r-', linewidth=2)
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('Velocity (km/s)')
ax.set_title('Velocity vs Time')
ax.grid(True, alpha=0.3)

# Altitude vs velocity (reentry profile)
ax = axes[1, 0]
ax.plot(velocities_kmps, altitudes_km, 'g-', linewidth=2)
ax.scatter([velocities_kmps[0]], [altitudes_km[0]], c='green', s=100, label='Start', zorder=5)
ax.scatter([velocities_kmps[-1]], [altitudes_km[-1]], c='red', s=100, label='End', zorder=5)
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Velocity (km/s)')
ax.set_ylabel('Altitude (km)')
ax.set_title('Altitude-Velocity Profile')
ax.legend()
ax.grid(True, alpha=0.3)

# Heat flux during reentry
ax = axes[1, 1]
heat_fluxes = []
for alt, v in zip(altitudes_km, velocities_kmps):
    if 40 <= alt <= 150:
        q = atm.heat_flux(alt, v * 1000, 'earth') / 1e6  # MW/m²
    else:
        q = 0
    heat_fluxes.append(q)

ax.plot(times_min, heat_fluxes, 'orange', linewidth=2)
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('Heat Flux (MW/m²)')
ax.set_title('Convective Heat Flux')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print("\nReentry Summary:")
print("=" * 50)
print(f"Initial altitude: {altitudes_km[0]:.0f} km")
print(f"Initial velocity: {velocities_kmps[0]:.2f} km/s")
print(f"Final altitude: {altitudes_km[-1]:.0f} km")
print(f"Final velocity: {velocities_kmps[-1]:.2f} km/s")
print(f"Peak heat flux: {max(heat_fluxes):.2f} MW/m²")
print(f"Total reentry time: {times_min[-1]:.1f} minutes")

## 6. Compare Earth vs Mars Entry

Mars has a much thinner atmosphere, requiring different entry strategies.

In [ ]:
# Entry conditions for each planet
entry_conditions = {
    'earth': {'alt': 120, 'v': 7.8},   # LEO deorbit
    'mars barycenter': {'alt': 125, 'v': 5.5},  # Mars orbit velocity
}

# Calculate entry profiles for each
altitudes = np.linspace(0, 130, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Drag deceleration comparison
ax = axes[0]
for body, cond in entry_conditions.items():
    name = body.replace(' barycenter', '').title()
    v_mps = cond['v'] * 1000
    decels = []
    for alt in altitudes:
        rho = atm.density(body, alt)
        F_drag = 0.5 * rho * v_mps**2 * Cd * area
        a = F_drag / mass / g0  # in g's
        decels.append(a)
    ax.semilogy(altitudes, decels, label=name, linewidth=2)

ax.axhline(y=10, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Altitude (km)')
ax.set_ylabel('Deceleration (g)')
ax.set_title('Entry Deceleration Comparison')
ax.legend()
ax.grid(True, alpha=0.3, which='both')
ax.set_ylim(1e-6, 1e2)

# Heat flux comparison
ax = axes[1]
for body, cond in entry_conditions.items():
    name = body.replace(' barycenter', '').title()
    v_mps = cond['v'] * 1000
    heat = [atm.heat_flux(alt, v_mps, body) / 1e6 for alt in altitudes]  # MW/m²
    ax.plot(altitudes, heat, label=name, linewidth=2)

ax.set_xlabel('Altitude (km)')
ax.set_ylabel('Heat Flux (MW/m²)')
ax.set_title('Entry Heat Flux Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Insight:")
print("Mars' thin atmosphere means less aerodynamic braking.")
print("EDL (Entry, Descent, Landing) at Mars requires supplemental")
print("propulsion or parachutes to slow down sufficiently.")

## Summary

Key takeaways from this reentry analysis:
1. **Multi-layer atmosphere models** provide accurate density profiles
2. **Heat flux scales with v³** - higher entry velocities dramatically increase heating
3. **Peak heating** occurs at ~60-80 km altitude for Earth entry
4. **Mars entry** is challenging due to thin atmosphere - less braking available
5. **SOI-based detection** automatically determines which atmosphere to use